<a href="https://colab.research.google.com/github/pacheco110798/DataAnalysis/blob/main/Practica2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files

df = pd.read_json("transacciones_ecommerce.json")
object_columns = ["cliente", "detalles_pago"]

def limpiar_formato_moneda(x):

  x = str(x).strip().replace("$", "").replace(" ", "")

  # find last separator
  last_dot = x.rfind(".")
  last_comma = x.rfind(",")

  # decide decimal separator (rightmost wins)
  decimal_pos = max(last_dot, last_comma)

  if decimal_pos == -1:
      # no separator at all
      return x

  integer_part = x[:decimal_pos]
  decimal_part = x[decimal_pos+1:]

  # remove ALL separators from integer part
  integer_part = integer_part.replace(".", "").replace(",", "")

  return integer_part + "." + decimal_part



for col in object_columns:
  expanded = pd.json_normalize(df[col]).add_prefix(f'{col}.')
  df = df.drop(columns=[col]).join(expanded)

df = df.rename(columns={
    'cliente.nombre': 'nombre_cliente',
    'cliente.ubicacion.ciudad': 'ciudad_cliente',
    'cliente.ubicacion.pais': 'pais_cliente',
    'detalles_pago.monto': 'monto_pago',
    'detalles_pago.metodo': 'metodo_pago',
})

df["monto_pago"] = df["monto_pago"].apply(limpiar_formato_moneda)
df["nombre_cliente"] = df["nombre_cliente"].str.strip().str.lower().str.title()

df["fecha"] = pd.to_datetime(df["fecha"], format="mixed", errors="coerce")
df["fecha"] = df["fecha"].dt.strftime("%d/%m/%Y")

df = df[df["fecha"].notna()]

df["items"] = pd.to_numeric(df["items"], errors="coerce").astype("Int64")

df["monto_pago"] = pd.to_numeric(df["monto_pago"], errors="coerce")
df["monto_por_item"] = df["monto_pago"] / df["items"]
df["monto_por_item"] = pd.to_numeric(df["monto_por_item"], errors="coerce").round(2)

df["zscore"] = (df["monto_pago"] - df["monto_pago"].mean()) / df["monto_pago"].std()

df.to_csv("ventas_final_limpio.csv", index=False, encoding="utf-8")
df.head()

,id_transaccion,fecha,items,nombre_cliente,ciudad_cliente,pais_cliente,monto_pago,metodo_pago,monto_por_item,zscore
0,TRX-1001,01/05/2024,2,Ana Martínez,Guadalajara,México,2500.8,Efectivo,1250.4,-0.567028
1,TRX-1002,05/02/2024,4,Juan Pérez,Monterrey,México,150.0,Efectivo,37.5,-1.243138
2,TRX-1003,03/05/2024,4,Ana Martínez,Mérida,México,150.0,Efectivo,37.5,-1.243138
3,TRX-1004,05/02/2024,2,Ana Martínez,Monterrey,México,2500.8,Transferencia,1250.4,-0.567028
5,TRX-1006,05/02/2024,3,María García,CDMX,México,5000.0,Tarjeta,1666.67,0.151763
